# Redshift → Databricks Unity Catalog Permission Sync (v1)

## Purpose
Extract **real** access-control policies (GRANTs) from Amazon Redshift system views and
generate the equivalent **Unity Catalog** GRANT statements for a **Lakehouse Federation
foreign catalog**, for human review before applying.

## Flow
1. **Configure** – connection + target settings (credentials pulled from a secret scope)
2. **Preflight** – verify the foreign catalog exists and Redshift is reachable *before* doing work
3. **Extract** – query Redshift system views (`SVV_DATABASE_PRIVILEGES`, `SVV_SCHEMA_PRIVILEGES`, `SVV_RELATION_PRIVILEGES`)
4. **Discover** – `SHOW SCHEMAS` on the foreign catalog to catch schemas without explicit grants
5. **Transform** – map Redshift privileges & principals (1:1 by name) → Unity Catalog grants
6. **Generate** – produce ready-to-run UC GRANT statements + export to a `.sql` file for review
7. **Apply** *(opt-in)* – execute the grants (off by default)

## Scope (as configured for SWA)
| Redshift | → | Unity Catalog |
|---|---|---|
| database grant | → | `GRANT USE CATALOG` |
| schema `USAGE` | → | `GRANT USE SCHEMA` |
| table/view `SELECT` | → | `GRANT SELECT ON TABLE` |

Write privileges (`INSERT`/`UPDATE`/`DELETE`/`CREATE`) are **intentionally skipped** — federated
foreign catalogs are read-only in UC.

## Prerequisites (must be true or the notebook will stop at preflight)
- The Lakehouse Federation foreign catalog exists and this compute can see it.
- Network path from the cluster to Redshift on port 5439 (security group / VPC / PrivateLink).
- Redshift user with access to system views (superuser recommended for full visibility).
- Target Databricks principals already exist in the account (1:1 name match, or add overrides).

---
**Author:** Dipankar Kushari · **Version:** 1.0

In [0]:
# =============================================================
# TEST SETUP ONLY  \u2014  NOT executed by this notebook.
# =============================================================
# These are REDSHIFT SQL statements used to seed test privileges so you can
# verify the end-to-end sync (extract -> generate -> apply). Run them in the
# Redshift Query Editor against database `oetrta` (NOT in this notebook cell).
# They are commented out on purpose. Remove after testing (see REVOKE block).
#
# --- Option A: grant to PUBLIC (maps to UC `account users`, always applies) ---
# GRANT USAGE ON SCHEMA amenon TO PUBLIC;
# GRANT SELECT ON ALL TABLES IN SCHEMA amenon TO PUBLIC;
#
# --- Option B: grant to a GROUP (maps 1:1 to a Databricks account group) ---
# CREATE GROUP swa_analysts;
# CREATE USER swa_analyst PASSWORD 'Analyst#2026' IN GROUP swa_analysts;   -- optional
# GRANT USAGE ON SCHEMA amenon                TO GROUP swa_analysts;
# GRANT SELECT ON amenon.crew_data            TO GROUP swa_analysts;
# GRANT SELECT ON amenon.nyctaxi_yellow_100k  TO GROUP swa_analysts;
# GRANT SELECT ON ALL TABLES IN SCHEMA amenon TO GROUP swa_analysts;
#   NOTE: for the UC apply to succeed, a Databricks account group `swa_analysts`
#   must exist and be assigned to the workspace (or map it via PRINCIPAL_OVERRIDES).
#
# --- Verify in Redshift (should list your test grantee, identity_type user/group/public) ---
# SELECT namespace_name AS schema_name, relation_name, privilege_type, identity_name, identity_type
# FROM svv_relation_privileges
# WHERE namespace_name = 'amenon'
# ORDER BY 1, 2, 3;
#
# --- Cleanup after testing ---
# REVOKE SELECT ON ALL TABLES IN SCHEMA amenon FROM PUBLIC;
# REVOKE USAGE  ON SCHEMA amenon              FROM PUBLIC;
# REVOKE SELECT ON ALL TABLES IN SCHEMA amenon FROM GROUP swa_analysts;
# REVOKE USAGE  ON SCHEMA amenon              FROM GROUP swa_analysts;
# DROP USER swa_analyst;
# DROP GROUP swa_analysts;
print("Test-setup cell: reference only \u2014 run the Redshift GRANTs in the Redshift Query Editor, not here.")

Test-setup cell: reference only — run the Redshift GRANTs in the Redshift Query Editor, not here.


In [0]:
%pip install redshift_connector -q
dbutils.library.restartPython()


Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


## 🔌 Quick Redshift connection test (run this first)

Before running the full sync, use the cell below to confirm the cluster can actually reach
Redshift and the credentials work. Credentials are read from the **secret scope**
`swa_redshift` (keys `host`/`user`/`password`) — nothing is hardcoded. Just run this cell.
If it fails, the rest of the notebook cannot work — fix the connection first (usually the
network path to `:5439`). Secret values are shown as `[REDACTED]` in output by design.

In [0]:
# =============================================================
# QUICK REDSHIFT CONNECTION TEST  \u2014  run this first, on its own
# =============================================================
import redshift_connector

# Credentials come from the secret scope `swa_redshift` (no plaintext in the notebook).
TEST_SCOPE    = "swa_redshift"
TEST_HOST     = dbutils.secrets.get(TEST_SCOPE, "host")
TEST_PORT     = 5439
TEST_DATABASE = "oetrta"
TEST_USER     = dbutils.secrets.get(TEST_SCOPE, "user")
TEST_PASSWORD = dbutils.secrets.get(TEST_SCOPE, "password")

print(f"Connecting to Redshift:{TEST_PORT}/{TEST_DATABASE} using secret scope `{TEST_SCOPE}` ...")
try:
    _conn = redshift_connector.connect(
        host=TEST_HOST, port=TEST_PORT, database=TEST_DATABASE,
        user=TEST_USER, password=TEST_PASSWORD, timeout=15,
    )
    _cur = _conn.cursor()
    _cur.execute("SELECT current_user, current_database(), version()")
    _u, _d, _v = _cur.fetchone()
    _conn.close()
    print("\u2705 SUCCESS \u2014 Redshift is reachable and credentials work.")
    print(f"   user     : {_u}")
    print(f"   database : {_d}")
    print(f"   version  : {_v}")
except Exception as e:
    print(f"\u274c FAILED: {str(e)[:300]}")
    print("\nCommon causes:")
    print("  \u2022 'connection timed out'  -> no network path to Redshift:5439 "
          "(security group / VPC peering / PrivateLink). This is the #1 blocker.")
    print("  \u2022 auth error             -> wrong user / password.")
    print("  \u2022 unknown host           -> wrong host or DNS not resolvable from the cluster.")

Connecting to Redshift:5439/oetrta using secret scope `swa_redshift` ...
✅ SUCCESS — Redshift is reachable and credentials work.
   user     : [REDACTED]
   database : oetrta
   version  : PostgreSQL 8.0.2 on i686-pc-linux-gnu, compiled by GCC gcc (GCC) 3.4.2 20041017 (Red Hat 3.4.2-6.fc3), Redshift 1.0.117891 


In [0]:
# =============================================================
# CONFIGURATION
# =============================================================
# Redshift credentials are read from the secret scope `swa_redshift`
# (keys host/user/password). No plaintext credentials in this notebook.
# The scope was created via:
#   databricks secrets create-scope swa_redshift
#   databricks secrets put-secret   swa_redshift host
#   databricks secrets put-secret   swa_redshift user
#   databricks secrets put-secret   swa_redshift password

dbutils.widgets.dropdown("apply_grants", "False", ["True", "False"], "Apply grants (False = review only)")
dbutils.widgets.text("secret_scope", "swa_redshift", "Secret scope name")
dbutils.widgets.text("uc_foreign_catalog", "dkushari_swa_rs", "UC foreign catalog")
dbutils.widgets.text("redshift_database", "oetrta", "Redshift database")
dbutils.widgets.text("output_path",
    "/Workspace/Users/dipankar.kushari@databricks.com/swa/Redshift-policy-sync/redshift_grants.sql",
    "Output .sql path")

APPLY_GRANTS       = dbutils.widgets.get("apply_grants") == "True"
SECRET_SCOPE       = dbutils.widgets.get("secret_scope")
UC_FOREIGN_CATALOG = dbutils.widgets.get("uc_foreign_catalog")
REDSHIFT_DATABASE  = dbutils.widgets.get("redshift_database")
OUTPUT_PATH        = dbutils.widgets.get("output_path")
REDSHIFT_PORT      = 5439

def _secret(scope, key):
    """Read a secret; return None if the scope/key is absent."""
    try:
        return dbutils.secrets.get(scope=scope, key=key)
    except Exception:
        return None

# All Redshift credentials come from the secret scope (no plaintext in this notebook).
REDSHIFT_HOST     = _secret(SECRET_SCOPE, "host")
REDSHIFT_USER     = _secret(SECRET_SCOPE, "user")
REDSHIFT_PASSWORD = _secret(SECRET_SCOPE, "password")
_creds_ok         = all([REDSHIFT_HOST, REDSHIFT_USER, REDSHIFT_PASSWORD])

# -------------------------------------------------------------
# Principal mapping (identity is 1:1 by NAME by default).
# Add entries here ONLY for principals whose UC name differs from
# their Redshift name. `public` maps to the built-in `account users`.
# -------------------------------------------------------------
PRINCIPAL_OVERRIDES = {
    # "redshift_name": "databricks_group_or_email",
    # "some_service_account": "svc-analytics@databricks.com",
}
MAP_PUBLIC_TO_ACCOUNT_USERS = True   # Redshift PUBLIC -> UC `account users`

# -------------------------------------------------------------
# Optional schema scope. Leave EMPTY to sync ALL user schemas (the default,
# always the behaviour when this list is empty). Populate with schema names
# to restrict the sync to just those, e.g.:
#     INCLUDE_SCHEMAS = ["amenon", "finance_schema"]
# System schemas are always excluded regardless.
# -------------------------------------------------------------
INCLUDE_SCHEMAS = []

# -------------------------------------------------------------
# Optionally also capture IMPLICIT (owner) privileges. Redshift object owners
# have implicit USAGE/SELECT that do NOT appear in the SVV_*_PRIVILEGES views.
# When True, each schema/table owner is added as a grantee (mapped 1:1 by name
# like any other principal). Default False, because owners are often the admin
# user whose matching UC principal may not exist (those grants would fail on
# apply). Set True to include them.
# -------------------------------------------------------------
CAPTURE_OWNER_PRIVILEGES = False

# -------------------------------------------------------------
# Optional EFFECTIVE-privileges mode (the most complete option). When True, in
# addition to explicit grants the notebook resolves each USER's *effective*
# access — explicit + ALL implicit paths (ownership, group/role membership,
# PUBLIC) — using Redshift's HAS_SCHEMA_PRIVILEGE / HAS_TABLE_PRIVILEGE.
# Trade-offs:
#   - can be slow (evaluates users x objects), and
#   - it produces per-USER grants, so the matching UC *users* must exist for the
#     apply to succeed (group mirroring no longer needed, but users must map 1:1).
# EFFECTIVE_EXCLUDE_SUPERUSERS skips superusers (they implicitly have everything,
# so you don't want to grant SELECT-on-everything to admins).
# -------------------------------------------------------------
EFFECTIVE_PRIVILEGES_MODE = False
EFFECTIVE_EXCLUDE_SUPERUSERS = True

_creds_msg = (f"loaded from secret scope `{SECRET_SCOPE}`" if _creds_ok
              else f"MISSING - check secret scope `{SECRET_SCOPE}` keys host/user/password")

print(f"Mode              : {'APPLY' if APPLY_GRANTS else 'REVIEW ONLY (no changes)'}")
print(f"Foreign catalog   : {UC_FOREIGN_CATALOG}")
print(f"Redshift          : (host from secret scope `{SECRET_SCOPE}`):{REDSHIFT_PORT}/{REDSHIFT_DATABASE}")
print(f"Credentials       : {_creds_msg}")
print(f"Output .sql       : {OUTPUT_PATH}")

Mode              : APPLY
Foreign catalog   : dkushari_swa_rs
Redshift          : (host from secret scope `swa_redshift`):5439/oetrta
Credentials       : loaded from secret scope `swa_redshift`
Output .sql       : /Workspace/Users/dipankar.kushari@databricks.com/swa/Redshift-policy-sync/redshift_grants.sql


In [0]:
# =============================================================
# Redshift connection helper (pure Python, no JDBC jar)
# =============================================================
import redshift_connector
import pandas as pd

def query_redshift(sql, description="", timeout=30):
    """Run a query against Redshift, return a pandas DataFrame (or None on failure)."""
    conn = None
    try:
        conn = redshift_connector.connect(
            host=REDSHIFT_HOST, port=REDSHIFT_PORT, database=REDSHIFT_DATABASE,
            user=REDSHIFT_USER, password=REDSHIFT_PASSWORD, timeout=timeout,
        )
        cur = conn.cursor()
        cur.execute(sql)
        cols = [d[0] for d in cur.description]
        df = pd.DataFrame(cur.fetchall(), columns=cols)
        if description:
            print(f"  \u2705 {description}: {len(df)} rows")
        return df
    except Exception as e:
        if description:
            print(f"  \u274c {description}: {str(e)[:200]}")
        return None
    finally:
        if conn is not None:
            try: conn.close()
            except Exception: pass

In [0]:
# =============================================================
# PREFLIGHT — fail fast with a clear message if prerequisites aren't met
# =============================================================
class PreflightError(Exception):
    pass

problems = []

# 1) Foreign catalog must exist and be visible to this compute
print("Preflight 1/3 \u2014 foreign catalog visibility ...")
try:
    cats = {r["catalog"] for r in spark.sql("SHOW CATALOGS").collect()}
    if UC_FOREIGN_CATALOG in cats:
        print(f"  \u2705 Catalog `{UC_FOREIGN_CATALOG}` is visible.")
    else:
        problems.append(
            f"Foreign catalog `{UC_FOREIGN_CATALOG}` was NOT found by this compute. "
            f"Create the Lakehouse Federation catalog first (or fix the name / grant the "
            f"running principal USE CATALOG). Visible catalogs: {sorted(cats)[:20]}"
        )
        print(f"  \u274c Catalog `{UC_FOREIGN_CATALOG}` not visible.")
except Exception as e:
    problems.append(f"Could not list catalogs: {e}")

# 2) Redshift password must be present
print("Preflight 2/3 \u2014 Redshift credentials ...")
if not REDSHIFT_PASSWORD:
    problems.append(
        f"No Redshift password. Put it in secret scope `{SECRET_SCOPE}` key `password` "
        "(create it with `databricks secrets put-secret {SECRET_SCOPE} password`)."
    )
    print("  \u274c Password missing.")
else:
    print("  \u2705 Password present.")

# 3) Redshift must be reachable
print("Preflight 3/3 \u2014 Redshift connectivity ...")
_conn_ok = False
if REDSHIFT_PASSWORD:
    t = query_redshift("SELECT current_user AS u, current_database() AS d", "", timeout=15)
    if t is not None and len(t):
        _conn_ok = True
        print(f"  \u2705 Connected as {t.iloc[0]['u']} to {t.iloc[0]['d']}.")
    else:
        problems.append(
            "Cannot reach Redshift on "
            f"{REDSHIFT_HOST}:{REDSHIFT_PORT}. Open the network path from this cluster "
            "(security group / VPC peering / PrivateLink on port 5439)."
        )
        print("  \u274c Redshift not reachable.")

print()
if problems:
    print("\u26d4 PREFLIGHT FAILED \u2014 resolve these before continuing:\n")
    for i, p in enumerate(problems, 1):
        print(f"  {i}. {p}\n")
    raise PreflightError("Preflight checks failed; see messages above.")
else:
    print("\u2705 All preflight checks passed.")

Preflight 1/3 — foreign catalog visibility ...
  ✅ Catalog `dkushari_swa_rs` is visible.
Preflight 2/3 — Redshift credentials ...
  ✅ Password present.
Preflight 3/3 — Redshift connectivity ...
  ✅ Connected as [REDACTED] to oetrta.

✅ All preflight checks passed.


In [0]:
# =============================================================
# STEP 1: Extract privileges from Redshift system views
# =============================================================
# Scope (applied in SQL):
#   - target database only (REDSHIFT_DATABASE)
#   - schemas: INCLUDE_SCHEMAS if that list is populated, else ALL user schemas
#     (system schemas are always excluded)
SYSTEM_SCHEMAS = ["information_schema", "pg_catalog", "pg_internal", "pg_automv", "catalog_history"]
_sys_in = ", ".join("'" + s + "'" for s in SYSTEM_SCHEMAS)

def _schema_pred(col):
    """SQL predicate limiting <col> to the chosen schemas (INCLUDE_SCHEMAS or all user schemas)."""
    if INCLUDE_SCHEMAS:
        vals = ", ".join("'" + s + "'" for s in INCLUDE_SCHEMAS)
        return f"{col} IN ({vals})"
    return f"{col} NOT IN ({_sys_in})"

_scope_msg = f"schemas {INCLUDE_SCHEMAS}" if INCLUDE_SCHEMAS else "ALL user schemas"
print(f"Extracting privileges from Redshift database `{REDSHIFT_DATABASE}` ({_scope_msg}) ...")

df_db_privs = query_redshift(f"""
    SELECT database_name, privilege_type, identity_name, identity_type, admin_option
    FROM svv_database_privileges
    WHERE database_name = '{REDSHIFT_DATABASE}'
    ORDER BY database_name, privilege_type, identity_name
""", "SVV_DATABASE_PRIVILEGES")

df_schema_privs = query_redshift(f"""
    SELECT namespace_name AS schema_name, privilege_type, identity_name, identity_type, admin_option
    FROM svv_schema_privileges
    WHERE {_schema_pred('namespace_name')}
    ORDER BY namespace_name, privilege_type, identity_name
""", "SVV_SCHEMA_PRIVILEGES")

df_table_privs = query_redshift(f"""
    SELECT namespace_name AS schema_name, relation_name, privilege_type, identity_name, identity_type, admin_option
    FROM svv_relation_privileges
    WHERE {_schema_pred('namespace_name')}
    ORDER BY namespace_name, relation_name, privilege_type, identity_name
""", "SVV_RELATION_PRIVILEGES")

# --- IMPLICIT (owner) privileges ---------------------------------------------
# Owners have implicit USAGE (schema) / SELECT (table/view) that the SVV_* views
# do NOT report. When enabled, add owners as synthetic grantees so they carry
# through the same mapping as explicit grants.
if CAPTURE_OWNER_PRIVILEGES:
    print("Capturing implicit owner privileges ...")
    df_sch_own = query_redshift(f"""
        SELECT nspname AS schema_name, pg_catalog.pg_get_userbyid(nspowner) AS identity_name
        FROM pg_namespace
        WHERE {_schema_pred('nspname')}
    """, "SCHEMA OWNERS (implicit)")

    df_tbl_own = query_redshift(f"""
        SELECT schemaname AS schema_name, tablename AS relation_name, tableowner AS identity_name
        FROM pg_tables
        WHERE {_schema_pred('schemaname')}
        UNION ALL
        SELECT schemaname AS schema_name, viewname AS relation_name, viewowner AS identity_name
        FROM pg_views
        WHERE {_schema_pred('schemaname')}
    """, "TABLE/VIEW OWNERS (implicit)")

    if df_sch_own is not None and len(df_sch_own):
        a = df_sch_own.copy()
        a["privilege_type"] = "USAGE"; a["identity_type"] = "user"; a["admin_option"] = False
        c = ["schema_name", "privilege_type", "identity_name", "identity_type", "admin_option"]
        df_schema_privs = (a[c] if df_schema_privs is None else
                           pd.concat([df_schema_privs, a[c]], ignore_index=True)).drop_duplicates().reset_index(drop=True)

    if df_tbl_own is not None and len(df_tbl_own):
        a = df_tbl_own.copy()
        a["privilege_type"] = "SELECT"; a["identity_type"] = "user"; a["admin_option"] = False
        c = ["schema_name", "relation_name", "privilege_type", "identity_name", "identity_type", "admin_option"]
        df_table_privs = (a[c] if df_table_privs is None else
                          pd.concat([df_table_privs, a[c]], ignore_index=True)).drop_duplicates().reset_index(drop=True)

# --- EFFECTIVE privileges (opt-in): resolve real per-user access -------------
# Uses HAS_SCHEMA_PRIVILEGE / HAS_TABLE_PRIVILEGE to expand explicit + implicit
# access (ownership, group/role membership, PUBLIC) into per-USER grant rows.
if EFFECTIVE_PRIVILEGES_MODE:
    print("Resolving EFFECTIVE privileges via has_*_privilege (can be slow) ...")
    _super = "AND u.usesuper = false" if EFFECTIVE_EXCLUDE_SUPERUSERS else ""

    df_eff_sch = query_redshift(f"""
        SELECT u.usename AS identity_name, n.nspname AS schema_name
        FROM pg_user u
        CROSS JOIN pg_namespace n
        WHERE {_schema_pred('n.nspname')}
          {_super}
          AND has_schema_privilege(u.usename, n.nspname, 'USAGE')
    """, "EFFECTIVE SCHEMA USAGE")

    df_eff_tbl = query_redshift(f"""
        SELECT u.usename AS identity_name, t.schema_name, t.relation_name
        FROM pg_user u
        CROSS JOIN (
            SELECT schemaname AS schema_name, tablename AS relation_name FROM pg_tables
            WHERE {_schema_pred('schemaname')}
            UNION ALL
            SELECT schemaname AS schema_name, viewname AS relation_name FROM pg_views
            WHERE {_schema_pred('schemaname')}
        ) t
        WHERE 1=1 {_super}
          AND has_table_privilege(u.usename,
                quote_ident(t.schema_name) || '.' || quote_ident(t.relation_name), 'SELECT')
    """, "EFFECTIVE TABLE SELECT")

    if df_eff_sch is not None and len(df_eff_sch):
        a = df_eff_sch.copy()
        a["privilege_type"] = "USAGE"; a["identity_type"] = "user"; a["admin_option"] = False
        c = ["schema_name", "privilege_type", "identity_name", "identity_type", "admin_option"]
        df_schema_privs = (a[c] if df_schema_privs is None else
                           pd.concat([df_schema_privs, a[c]], ignore_index=True)).drop_duplicates().reset_index(drop=True)

    if df_eff_tbl is not None and len(df_eff_tbl):
        a = df_eff_tbl.copy()
        a["privilege_type"] = "SELECT"; a["identity_type"] = "user"; a["admin_option"] = False
        c = ["schema_name", "relation_name", "privilege_type", "identity_name", "identity_type", "admin_option"]
        df_table_privs = (a[c] if df_table_privs is None else
                          pd.concat([df_table_privs, a[c]], ignore_index=True)).drop_duplicates().reset_index(drop=True)

# Reference only: users & groups (helps you spot unmapped principals)
df_users = query_redshift("""
    SELECT u.usename AS username, u.usesuper AS is_superuser, g.groname AS group_name
    FROM pg_user u
    LEFT JOIN pg_group g ON u.usesysid = ANY(g.grolist)
    ORDER BY u.usename
""", "PG_USER + PG_GROUP")

print("\nExtraction complete.")

Extracting privileges from Redshift database `oetrta` (ALL user schemas) ...
  ✅ SVV_DATABASE_PRIVILEGES: 1 rows
  ✅ SVV_SCHEMA_PRIVILEGES: 4 rows
  ✅ SVV_RELATION_PRIVILEGES: 6 rows
  ✅ PG_USER + PG_GROUP: 5 rows

Extraction complete.


In [0]:
# =============================================================
# STEP 2: Inspect the raw Redshift privileges (nothing fabricated)
# =============================================================
def show(df, title):
    print(f"\n--- {title} ---")
    if df is not None and len(df):
        print(df.to_string(index=False))
    else:
        print("(no rows / extraction failed)")

show(df_db_privs,     "DATABASE PRIVILEGES")
show(df_schema_privs, "SCHEMA PRIVILEGES")
show(df_table_privs,  "TABLE/VIEW PRIVILEGES")
show(df_users,        "USERS & GROUPS (reference)")


--- DATABASE PRIVILEGES ---
database_name privilege_type identity_name identity_type  admin_option
       oetrta           TEMP        public        public         False

--- SCHEMA PRIVILEGES ---
schema_name privilege_type identity_name identity_type  admin_option
     amenon          USAGE        public        public         False
     amenon          USAGE  swa_analysts         group         False
     public         CREATE        public        public         False
     public          USAGE        public        public         False

--- TABLE/VIEW PRIVILEGES ---
schema_name                        relation_name privilege_type identity_name identity_type  admin_option
     amenon                            crew_data         SELECT        public        public         False
     amenon                            crew_data         SELECT  swa_analysts         group         False
     amenon                  nyctaxi_yellow_100k         SELECT        public        public         False
  

In [0]:
# =============================================================
# STEP 3: Discover schemas present in the foreign catalog
# =============================================================
all_schema_names = []
try:
    rows = spark.sql(f"SHOW SCHEMAS IN `{UC_FOREIGN_CATALOG}`").collect()
    all_schema_names = [r["databaseName"] for r in rows if r["databaseName"] != "information_schema"]
    if INCLUDE_SCHEMAS:
        all_schema_names = [s for s in all_schema_names if s in INCLUDE_SCHEMAS]
    print(f"\u2705 {len(all_schema_names)} schemas in `{UC_FOREIGN_CATALOG}`:")
    for s in sorted(all_schema_names):
        print(f"    - {s}")
except Exception as e:
    print(f"\u26a0\ufe0f  Could not list schemas in `{UC_FOREIGN_CATALOG}`: {str(e)[:200]}")
    print("    Continuing with only the schemas that have explicit Redshift grants.")

✅ 37 schemas in `dkushari_swa_rs`:
    - amenon
    - amullins_demo
    - athena_delta
    - catalog_history
    - databricks_rs_dk
    - dbt_demo
    - depop_external
    - finance_schema
    - keerthi_josyula
    - keerthi_josyula_external
    - keerthi_josyula_external2
    - migrations_test
    - migrations_test_spectrum
    - migrations_test_spectrum_2
    - nyl_test
    - pg_auto_copy
    - pg_automv
    - pg_catalog
    - pg_internal
    - pg_mv
    - pg_s3
    - profiler
    - public
    - sewi_spectrum
    - sewi_spectrum_oetrta
    - sewi_spectrum_schema
    - smunigati_external_spectrum
    - smunigati_tpcds
    - smunigati_tpcds_test
    - spaladugu_crisp_retail_and_distributor
    - suncorp_demo
    - telco
    - tenbosch_test
    - tpcdi
    - tpch_sf1
    - vadim_patsalo
    - zhanf_demo


In [0]:
# =============================================================
# STEP 4: Transform Redshift privileges -> Unity Catalog GRANTs
# =============================================================
# Scope: catalog USE CATALOG, schema USE SCHEMA, table SELECT.
# Identity: 1:1 by name, with PRINCIPAL_OVERRIDES and public->account users.

def quote_principal(name):
    """Backtick-quote a UC principal (required for names with spaces, e.g. `account users`)."""
    return "`" + name.replace("`", "``") + "`"

def map_principal(identity_name, identity_type):
    """Return the UC principal for a Redshift identity, or None to skip."""
    if identity_name in PRINCIPAL_OVERRIDES:
        return PRINCIPAL_OVERRIDES[identity_name]
    if (identity_type or "").lower() == "public" or (identity_name or "").lower() == "public":
        return "account users" if MAP_PUBLIC_TO_ACCOUNT_USERS else None
    # 1:1 by name
    return identity_name

grant_statements = []          # list of {sql, level, principal}
unmapped = set()
mapped_principals = set()

def add_grant(sql, level, principal):
    grant_statements.append({"sql": sql, "level": level, "principal": principal})
    mapped_principals.add(principal)

# --- Table/view SELECT ---
if df_table_privs is not None and len(df_table_privs):
    for _, r in df_table_privs.iterrows():
        if str(r["privilege_type"]).upper() != "SELECT":
            continue
        p = map_principal(r["identity_name"], r["identity_type"])
        if p is None:
            unmapped.add((r["identity_name"], r["identity_type"])); continue
        tbl = f"`{UC_FOREIGN_CATALOG}`.`{r['schema_name']}`.`{r['relation_name']}`"
        add_grant(f"GRANT SELECT ON TABLE {tbl} TO {quote_principal(p)};", "TABLE", p)

# --- Schema USAGE -> USE SCHEMA ---
if df_schema_privs is not None and len(df_schema_privs):
    for _, r in df_schema_privs.iterrows():
        if str(r["privilege_type"]).upper() != "USAGE":
            continue
        p = map_principal(r["identity_name"], r["identity_type"])
        if p is None:
            unmapped.add((r["identity_name"], r["identity_type"])); continue
        sch = f"`{UC_FOREIGN_CATALOG}`.`{r['schema_name']}`"
        add_grant(f"GRANT USE SCHEMA ON SCHEMA {sch} TO {quote_principal(p)};", "SCHEMA", p)

# --- Ensure USE SCHEMA for schemas that have SELECT grants but no explicit USAGE row ---
schemas_with_use = {g["sql"].split("ON SCHEMA `")[1].split("`")[1]
                    for g in grant_statements if g["level"] == "SCHEMA"}
for g in list(grant_statements):
    if g["level"] == "TABLE":
        schema = g["sql"].split(f"`{UC_FOREIGN_CATALOG}`.`")[1].split("`")[0]
        key = (schema, g["principal"])
        if schema not in schemas_with_use:
            sch = f"`{UC_FOREIGN_CATALOG}`.`{schema}`"
            stmt = f"GRANT USE SCHEMA ON SCHEMA {sch} TO {quote_principal(g['principal'])};"
            if stmt not in {x["sql"] for x in grant_statements}:
                add_grant(stmt, "SCHEMA", g["principal"])

# --- USE CATALOG for every mapped principal (prepended) ---
for p in sorted(mapped_principals):
    grant_statements.insert(0, {
        "sql": f"GRANT USE CATALOG ON CATALOG `{UC_FOREIGN_CATALOG}` TO {quote_principal(p)};",
        "level": "CATALOG", "principal": p,
    })

print(f"\u2705 Generated {len(grant_statements)} GRANT statements for {len(mapped_principals)} principals.")
if unmapped:
    print(f"\n\u26a0\ufe0f  {len(unmapped)} Redshift identities were skipped (no 1:1 UC match). "
          f"Add them to PRINCIPAL_OVERRIDES if they should be included:")
    for name, typ in sorted(unmapped):
        print(f"    - {name} (type={typ})")

✅ Generated 11 GRANT statements for 2 principals.


In [0]:
# =============================================================
# STEP 5: Review the generated GRANT statements
# =============================================================
print("=" * 80)
print(f"  GENERATED UC GRANTS FOR: {UC_FOREIGN_CATALOG}")
print("=" * 80)
for level in ("CATALOG", "SCHEMA", "TABLE"):
    block = [g["sql"] for g in grant_statements if g["level"] == level]
    if block:
        print(f"\n-- {level} ({len(block)}) --")
        for s in block:
            print(s)
if not grant_statements:
    print("(no grants generated \u2014 check extraction results and principal mapping)")

  GENERATED UC GRANTS FOR: dkushari_swa_rs

-- CATALOG (2) --
GRANT USE CATALOG ON CATALOG `dkushari_swa_rs` TO `swa_analysts`;
GRANT USE CATALOG ON CATALOG `dkushari_swa_rs` TO `account users`;

-- SCHEMA (3) --
GRANT USE SCHEMA ON SCHEMA `dkushari_swa_rs`.`amenon` TO `account users`;
GRANT USE SCHEMA ON SCHEMA `dkushari_swa_rs`.`amenon` TO `swa_analysts`;
GRANT USE SCHEMA ON SCHEMA `dkushari_swa_rs`.`public` TO `account users`;

-- TABLE (6) --
GRANT SELECT ON TABLE `dkushari_swa_rs`.`amenon`.`crew_data` TO `account users`;
GRANT SELECT ON TABLE `dkushari_swa_rs`.`amenon`.`crew_data` TO `swa_analysts`;
GRANT SELECT ON TABLE `dkushari_swa_rs`.`amenon`.`nyctaxi_yellow_100k` TO `account users`;
GRANT SELECT ON TABLE `dkushari_swa_rs`.`amenon`.`nyctaxi_yellow_100k` TO `swa_analysts`;
GRANT SELECT ON TABLE `dkushari_swa_rs`.`amenon`.`nyctaxi_yellow_100k_daily_revenue_vw` TO `account users`;
GRANT SELECT ON TABLE `dkushari_swa_rs`.`amenon`.`nyctaxi_yellow_100k_daily_revenue_vw` TO `swa_ana

In [0]:
# =============================================================
# STEP 6: Export grants to a .sql file for review
# =============================================================
import os
from datetime import datetime

header = [
    "-- Redshift -> Unity Catalog permission sync",
    f"-- Generated : {datetime.now():%Y-%m-%d %H:%M:%S}",
    f"-- Source     : {REDSHIFT_HOST}:{REDSHIFT_PORT}/{REDSHIFT_DATABASE}",
    f"-- Target     : {UC_FOREIGN_CATALOG}",
    f"-- Statements : {len(grant_statements)}",
    "",
]
sql_text = "\n".join(header + [g["sql"] for g in grant_statements]) + "\n"

def _write(path, text):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w") as f:
        f.write(text)

try:
    _write(OUTPUT_PATH, sql_text)
    print(f"\u2705 Exported {len(grant_statements)} statements to: {OUTPUT_PATH}")
except Exception as e:
    fallback = "/tmp/redshift_grants.sql"
    _write(fallback, sql_text)
    print(f"\u26a0\ufe0f  Could not write to {OUTPUT_PATH}: {str(e)[:150]}")
    print(f"    Saved to fallback: {fallback}")

✅ Exported 11 statements to: /Workspace/Users/dipankar.kushari@databricks.com/swa/Redshift-policy-sync/redshift_grants.sql


In [0]:
# =============================================================
# STEP 7: Apply grants (opt-in)  \u2014  set widget apply_grants=True
# =============================================================
if not APPLY_GRANTS:
    print(f"\U0001f7e2 REVIEW ONLY \u2014 {len(grant_statements)} grants NOT applied.")
    print("    Review the .sql output above, then set the `apply_grants` widget to True and re-run this cell.")
else:
    print(f"\U0001f534 APPLYING {len(grant_statements)} grants ...")
    ok = fail = 0
    for g in grant_statements:
        stmt = g["sql"].rstrip(";")
        try:
            spark.sql(stmt); ok += 1
            print(f"  \u2705 {stmt}")
        except Exception as e:
            fail += 1
            print(f"  \u274c {stmt}\n      {str(e)[:150]}")
    print(f"\nResults: {ok} succeeded, {fail} failed, {len(grant_statements)} total.")

🔴 APPLYING 11 grants ...
  ✅ GRANT USE CATALOG ON CATALOG `dkushari_swa_rs` TO `swa_analysts`
  ✅ GRANT USE CATALOG ON CATALOG `dkushari_swa_rs` TO `account users`
  ✅ GRANT SELECT ON TABLE `dkushari_swa_rs`.`amenon`.`crew_data` TO `account users`
  ✅ GRANT SELECT ON TABLE `dkushari_swa_rs`.`amenon`.`crew_data` TO `swa_analysts`
  ✅ GRANT SELECT ON TABLE `dkushari_swa_rs`.`amenon`.`nyctaxi_yellow_100k` TO `account users`
  ✅ GRANT SELECT ON TABLE `dkushari_swa_rs`.`amenon`.`nyctaxi_yellow_100k` TO `swa_analysts`
  ✅ GRANT SELECT ON TABLE `dkushari_swa_rs`.`amenon`.`nyctaxi_yellow_100k_daily_revenue_vw` TO `account users`
  ✅ GRANT SELECT ON TABLE `dkushari_swa_rs`.`amenon`.`nyctaxi_yellow_100k_daily_revenue_vw` TO `swa_analysts`
  ✅ GRANT USE SCHEMA ON SCHEMA `dkushari_swa_rs`.`amenon` TO `account users`
  ✅ GRANT USE SCHEMA ON SCHEMA `dkushari_swa_rs`.`amenon` TO `swa_analysts`
  ✅ GRANT USE SCHEMA ON SCHEMA `dkushari_swa_rs`.`public` TO `account users`

Results: 11 succeeded, 0 faile

In [0]:
%sql
SHOW GRANTS ON CATALOG `dkushari_swa_rs`;


Principal,ActionType,ObjectType,ObjectKey
finance_admin@databricks.com,USE CATALOG,CATALOG,dkushari_swa_rs
app_readonly@databricks.com,USE CATALOG,CATALOG,dkushari_swa_rs
etl_user@databricks.com,USE CATALOG,CATALOG,dkushari_swa_rs
data_science_team,USE CATALOG,CATALOG,dkushari_swa_rs
analyst_team,USE CATALOG,CATALOG,dkushari_swa_rs
swa_analysts,USE CATALOG,CATALOG,dkushari_swa_rs
account users,BROWSE,CATALOG,dkushari_swa_rs
account users,USE CATALOG,CATALOG,dkushari_swa_rs
57e7dc57-63b9-4114-a2ad-0a6226794a22,ALL PRIVILEGES,CATALOG,dkushari_swa_rs
57e7dc57-63b9-4114-a2ad-0a6226794a22,MANAGE,CATALOG,dkushari_swa_rs


In [0]:
%sql
SHOW GRANTS ON SCHEMA  `dkushari_swa_rs`.`amenon`;


Principal,ActionType,ObjectType,ObjectKey
swa_analysts,USE SCHEMA,SCHEMA,dkushari_swa_rs.amenon
dipankar.kushari@databricks.com,SELECT,SCHEMA,dkushari_swa_rs.amenon
dipankar.kushari@databricks.com,USE SCHEMA,SCHEMA,dkushari_swa_rs.amenon
account users,USE SCHEMA,SCHEMA,dkushari_swa_rs.amenon
57e7dc57-63b9-4114-a2ad-0a6226794a22,ALL PRIVILEGES,CATALOG,dkushari_swa_rs
57e7dc57-63b9-4114-a2ad-0a6226794a22,MANAGE,CATALOG,dkushari_swa_rs


In [0]:
%sql
SHOW GRANTS ON TABLE   `dkushari_swa_rs`.`amenon`.`crew_data`;


Principal,ActionType,ObjectType,ObjectKey
swa_analysts,SELECT,TABLE,dkushari_swa_rs.amenon.crew_data
dipankar.kushari@databricks.com,SELECT,SCHEMA,dkushari_swa_rs.amenon
account users,SELECT,TABLE,dkushari_swa_rs.amenon.crew_data
57e7dc57-63b9-4114-a2ad-0a6226794a22,ALL PRIVILEGES,CATALOG,dkushari_swa_rs
57e7dc57-63b9-4114-a2ad-0a6226794a22,MANAGE,CATALOG,dkushari_swa_rs
